In [ ]:
# Célula 1: Importações e utilitários
import pandas as pd
import numpy as np
from collections import Counter
import ast

# Bibliotecas de ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Tentativa de import XGBoost
try:
    from xgboost import XGBClassifier
    tem_xgb = True
except ImportError:
    tem_xgb = False

# Função para interpretar strings de listas ou split por vírgula
def parsear_lista(s):
    """
    Interpreta cada string como lista via ast.literal_eval ou faz split por vírgula.
    Retorna lista limpa de itens.
    """
    if pd.isna(s):
        return []
    try:
        valores = ast.literal_eval(s)
        if isinstance(valores, list):
            return [item.strip() for item in valores if item.strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in s.split(',') if item.strip()]

In [ ]:
# Célula 2: Carregar dados
df_treino = pd.read_csv('dados_clientes.csv')
df_teste  = pd.read_csv('desafio.csv')

In [ ]:
# Célula 3: Exploração inicial
display(df_treino.head())
df_treino.info()
print("Distribuição de churn (proporção):\n", df_treino['churn'].value_counts(normalize=True))

,id_cliente,idade,genero,estado_civil,tempo_como_cliente,tipo_contrato,forma_pagamento,suporte_contatado,chamados_abertos,tempo_medio_atendimento,reclamacoes,atrasos_pagamento,renda_faixa,servicos_assinados,produtos_assinados,valor_mensal,total_gasto,churn
0,c19f9c98-53b9-4332-bc13-5ef6140340bf,44,Feminino,Solteiro,53,Mensal,Cartão,3,0,27.13,0,1,5000-10000,1,['Produto B'],92.50,5008.27,0.0
1,31a56465-bba5-4d96-b469-e175557147b1,38,Masculino,Solteiro,108,Mensal,Cartão,0,0,25.33,1,2,10000-50000,2,['Produto A' 'Produto B'],154.26,17019.52,0.0
2,e35a2241-9ccd-4f48-9115-5fba0453f3c1,46,Feminino,Divorciado,56,Mensal,Boleto,0,1,27.43,2,3,1000-5000,1,['Produto E'],202.58,11589.24,0.0
3,db753c09-0ee7-4d19-907d-7407a9d9e45c,55,Feminino,Casado,110,Mensal,Cartão,1,2,11.50,2,3,5000-10000,1,['Produto F'],344.72,38737.30,0.0
4,d719f4aa-0bc2-4784-a5e6-dddaaf39b229,37,Masculino,Casado,17,Mensal,Boleto,2,2,21.42,0,2,5000-10000,5,['Produto C' 'Produto B' 'Produto D' 'Produto ...,871.10,15128.20,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6934 entries, 0 to 6933
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_cliente               6934 non-null   object 
 1   idade                    6934 non-null   int64  
 2   genero                   6934 non-null   object 
 3   estado_civil             6934 non-null   object 
 4   tempo_como_cliente       6934 non-null   int64  
 5   tipo_contrato            6934 non-null   object 
 6   forma_pagamento          6934 non-null   object 
 7   suporte_contatado        6934 non-null   int64  
 8   chamados_abertos         6934 non-null   int64  
 9   tempo_medio_atendimento  6934 non-null   float64
 10  reclamacoes              6934 non-null   int64  
 11  atrasos_pagamento        6934 non-null   int64  
 12  renda_faixa              6934 non-null   object 
 13  servicos_assinados       6934 non-null   int64  
 14  produtos_assinados      

In [ ]:
# Célula 4: Definição das listas de colunas
numericos_com_skew = ['valor_mensal', 'total_gasto', 'tempo_medio_atendimento']
numericos_contagem = [
    'idade', 'tempo_como_cliente', 'suporte_contatado',
    'chamados_abertos', 'reclamacoes', 'atrasos_pagamento', 'servicos_assinados'
]
categoricos_nominais = ['genero', 'estado_civil', 'tipo_contrato', 'forma_pagamento']
categoricos_ordinais  = ['renda_faixa']

# Ordem das categorias para variável ordinal
categorias_renda = [[
    '0-1000', '1000-5000', '5000-10000', '10000-50000', '50000-100000'
]]


In [ ]:
# Célula 5: Engenharia de features de produtos
# 5.1 - Extrair lista de produtos do treino
df_treino['lista_produtos'] = df_treino['produtos_assinados'].apply(parsear_lista)
lista_todos_produtos = [item for sub in df_treino['lista_produtos'] for item in sub]
produtos_top6 = [prod for prod, _ in Counter(lista_todos_produtos).most_common(6)]

# 5.2 - Função para adicionar colunas binárias de produtos
def adicionar_features_produtos(df, lista_produtos):
    df['lista_produtos'] = df['produtos_assinados'].apply(parsear_lista)
    for prod in lista_produtos:
        df[f'prod_{prod}'] = df['lista_produtos'].apply(lambda itens: int(prod in itens))
    df.drop(['produtos_assinados', 'lista_produtos'], axis=1, inplace=True)
    return df

# Aplica transformação ao treino e teste
df_treino = adicionar_features_produtos(df_treino, produtos_top6)
df_teste  = adicionar_features_produtos(df_teste, produtos_top6)

In [ ]:
# Célula 6: Construção dos pipelines de pré-processamento
# Pipeline para numéricos com skew: mediana, log1p, padronização
pipe_num_skew = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])
# Pipeline para numéricos de contagem: mediana, escala robusta
pipe_num_cont = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('robust', RobustScaler())
])
# Pipeline para ordinal: moda, codificador ordinal
pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=categorias_renda))
])
# Pipeline para nominais: preencher 'missing', one-hot
pipe_nom = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer reunindo todos os pipelines
preprocessor = ColumnTransformer([
    ('skew', pipe_num_skew, numericos_com_skew),
    ('cont', pipe_num_cont, numericos_contagem),
    ('ord',  pipe_ord,  categoricos_ordinais),
    ('nom',  pipe_nom, categoricos_nominais),
    # colunas de produtos já binárias: passthrough
    ('produtos', 'passthrough', [f'prod_{p}' for p in produtos_top6])
], remainder='drop')


In [ ]:
# Célula 7: Separação de X, y e transformação
y = df_treino['churn']
X = df_treino.drop(['id_cliente', 'churn'], axis=1)
X = X[~y.isna()]
y = y[~y.isna()]
X_teste = df_teste.drop(['id_cliente'], axis=1)

preprocessor.fit(X)
X_proc       = preprocessor.transform(X)
X_teste_proc = preprocessor.transform(X_teste)

In [ ]:
# Célula 8: Extração dos nomes das features
nomes_features = []
for nome, trans, cols in preprocessor.transformers_:
    if trans == 'passthrough':
        nomes_features += cols
    elif hasattr(trans, 'named_steps') and 'onehot' in trans.named_steps:
        ohe = trans.named_steps['onehot']
        nomes_features += ohe.get_feature_names_out(cols).tolist()
    else:
        nomes_features += cols
# Opcional: visualizar DataFrame transformado
df_transformado = pd.DataFrame(X_proc, columns=nomes_features)
display(df_transformado.head())


,valor_mensal,total_gasto,tempo_medio_atendimento,idade,tempo_como_cliente,suporte_contatado,chamados_abertos,reclamacoes,atrasos_pagamento,servicos_assinados,...,tipo_contrato_Mensal,forma_pagamento_Boleto,forma_pagamento_Cartão,forma_pagamento_Débito,prod_Produto A,prod_Produto B,prod_Produto E,prod_Produto C,prod_Produto D,prod_Produto F
0,-0.554654,-0.181885,1.144877,0.384615,-0.131148,0.5,-0.5,0.0,-0.5,-0.333333,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,-0.255591,0.256390,0.995191,-0.076923,0.770492,-1.0,-0.5,1.0,0.0,0.000000,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.095806,0.118702,1.168890,0.538462,-0.081967,-1.0,0.0,2.0,0.5,-0.333333,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.216484,0.551077,-0.691142,1.230769,0.803279,-0.5,0.5,2.0,0.5,-0.333333,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.762122,0.214182,0.631305,-0.153846,-0.721311,0.0,0.5,0.0,0.0,1.000000,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Célula 9: Treinamento e avaliação de modelo
from sklearn.model_selection import train_test_split

X_treino, X_val, y_treino, y_val = train_test_split(X_proc, y, test_size=0.2, random_state=42)

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_treino, y_treino)

y_pred_val = modelo_rf.predict(X_val)

print("Acurácia na validação:", accuracy_score(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

from sklearn.model_selection import cross_val_score

scores = cross_val_score(modelo_rf, X_proc, y, cv=5, scoring='accuracy')
print("Acurácia média (cross-val):", scores.mean())

Acurácia na validação: 0.7094448449891853
              precision    recall  f1-score   support

         0.0       0.74      0.87      0.80       921
         1.0       0.60      0.39      0.47       466

    accuracy                           0.71      1387
   macro avg       0.67      0.63      0.64      1387
weighted avg       0.69      0.71      0.69      1387

Acurácia média (cross-val): 0.7209024012917308


In [ ]:
# Célula 10: Comparação manual de múltiplos modelos
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier()
}

print("\nComparação de modelos com validação cruzada (5-fold):\n")
for nome, modelo in modelos.items():
    acc = cross_val_score(modelo, X_proc, y, cv=5, scoring='accuracy')
    print(f"{nome}: Acurácia média = {acc.mean():.4f} ± {acc.std():.4f}")

# Treinando o melhor modelo manualmente (Gradient Boosting)
melhor_modelo = GradientBoostingClassifier()
melhor_modelo.fit(X_proc, y)

# Prever no conjunto de teste
y_pred_final = melhor_modelo.predict(X_teste_proc)

# Gerar CSV de saída
df_resultado = pd.DataFrame({
    'id_cliente': df_teste['id_cliente'],
    'churn': y_pred_final
})


Comparação de modelos com validação cruzada (5-fold):

Logistic Regression: Acurácia média = 0.7186 ± 0.0064
Random Forest: Acurácia média = 0.7161 ± 0.0094


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [16:34:04] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [16:34:05] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [16:34:05] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [16:34:05] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [16:34:05] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

XGBoost: Acurácia média = 0.6929 ± 0.0085
Gradient Boosting: Acurácia média = 0.7196 ± 0.0108
KNN: Acurácia média = 0.6522 ± 0.0109
SVM: Acurácia média = 0.7154 ± 0.0069
Decision Tree: Acurácia média = 0.6044 ± 0.0035
\nArquivo 'resultado_seu_nome.csv' gerado com sucesso com o modelo Gradient Boosting.


In [ ]:
# Célula 11: Rebalanceamento de classes + Ajuste de Hiperparâmetros (tuning)
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

# Aplicar SMOTE para balancear as classes
over = SMOTE(random_state=42)
X_res, y_res = over.fit_resample(X_proc, y)
print("Classes balanceadas após SMOTE:", np.bincount(y_res))

# Dicionário de modelos e grids para testar
modelos_com_grid = {
    "Gradient Boosting": {
        'modelo': GradientBoostingClassifier(),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5]
        }
    },
    "Random Forest": {
        'modelo': RandomForestClassifier(),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20]
        }
    },
    "XGBoost": {
        'modelo': XGBClassifier(eval_metric='logloss'),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5]
        }
    }
}

# Rodar GridSearchCV para cada modelo e guardar os resultados
resultados_tuning = {}

for nome, cfg in modelos_com_grid.items():
    print(f"\n🔍 Ajustando hiperparâmetros para: {nome}")
    grid = GridSearchCV(cfg['modelo'], cfg['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_res, y_res)
    resultados_tuning[nome] = grid
    print("Melhores parâmetros:", grid.best_params_)
    print("Melhor acurácia (cross-val):", grid.best_score_)

# Escolher o melhor modelo baseado no score
melhor_nome = max(resultados_tuning, key=lambda n: resultados_tuning[n].best_score_)
melhor_grid = resultados_tuning[melhor_nome]

print(f"\n✅ Modelo final selecionado: {melhor_nome} com acurácia de {melhor_grid.best_score_:.4f}")

# Gerar previsões no conjunto de teste
y_pred_final = melhor_grid.predict(X_teste_proc)
df_resultado = pd.DataFrame({
    'id_cliente': df_teste['id_cliente'],
    'churn': y_pred_final
})
df_resultado.to_csv('resultado_tunado_seu_nome.csv', index=False)
print("\n📁 Arquivo 'resultado_tunado_seu_nome.csv' gerado com sucesso com o modelo tunado e rebalanceado.")


Classes balanceadas após SMOTE: [4622 4622]

🔍 Ajustando hiperparâmetros para: Gradient Boosting
Melhores parâmetros: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}
Melhor acurácia (cross-val): 0.7754358270177631

🔍 Ajustando hiperparâmetros para: Random Forest
Melhores parâmetros: {'max_depth': None, 'n_estimators': 200}
Melhor acurácia (cross-val): 0.7904693422676117

🔍 Ajustando hiperparâmetros para: XGBoost
Melhores parâmetros: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}
Melhor acurácia (cross-val): 0.7724064604946163

✅ Modelo final selecionado: Random Forest com acurácia de 0.7905

📁 Arquivo 'resultado_tunado_seu_nome.csv' gerado com sucesso com o modelo tunado e rebalanceado.
